# 🌿 Plant Watering AI — обучение дерева решений в Google Colab
**Источник:** только [`jibrand/Plant-dataset`](https://huggingface.co/datasets/jibrand/Plant-dataset).  
**Цель:** классифицировать **режим полива** (редкий / умеренный / регулярный) на основе названия растения и типа освещения. Датасет **не содержит измерений влажности почвы или даты следующего полива**, поэтому эти характеристики нельзя честно обучить на нём.

Процедура: загрузка → диагностика → очистка → кодирование (нормализация категорий) → train/test → Decision Tree → оценка → экспорт `joblib` для Streamlit.

## 1. Установка и импорт библиотек

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn huggingface_hub fsspec hf_xet

import os, json, joblib, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, ConfusionMatrixDisplay
warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 2. Загрузка CSV с Hugging Face (автоматический запасной адрес)

In [ ]:
HF_PATH = "hf://datasets/jibrand/Plant-dataset/plant_dataset.csv"
try:
    df = pd.read_csv(HF_PATH, low_memory=False)
    print('Данные загружены по hf://')
except Exception as e:
    print('hf:// недоступен:', str(e)[:250])
    df = pd.read_csv('https://huggingface.co/datasets/jibrand/Plant-dataset/resolve/main/plant_dataset.csv', low_memory=False)
    print('Данные загружены по HTTPS')
print('Размер:',df.shape)
display(df.head(10))

## 3. Первичный анализ: пропуски, дубли, классы и некорректные столбцы

In [ ]:
display(df.isna().sum().rename('Пропущено').to_frame())
print('Число полностью повторяющихся строк:',df.duplicated().sum())
print('Наиболее частые значения Watering:')
display(df['Watering'].astype(str).value_counts().head(20).to_frame('Количество'))
print('ВНИМАНИЕ: в исходном CSV есть смещения: в Watering встречаются виды почвы, а в Humidity — инструкции о поливе. Не используем такие строки в обучении.')

## 4. Очистка и нормализация категорий
Очистим кавычки, пробелы и регистр. Цель — **категориальный** режим полива; нормализация числовых датчиков здесь невозможна, так как их нет в наборе.

In [ ]:
TARGET_MAP = {
 'moderate':'Умеренный', 'regular':'Регулярный','regularly':'Регулярный','water regularly':'Регулярный',
 'infrequent':'Редкий','infrequently':'Редкий','water sparingly':'Редкий','water moderately':'Умеренный'
}
LIGHT_MAP = {
 'full sun':'Полное солнце','full sun to partial shade':'Солнце / полутень',
 'partial shade to full sun':'Солнце / полутень','partial shade':'Полутень',
 'bright indirect light':'Яркий рассеянный','low light':'Слабое освещение','shade':'Тень'
}
def norm(s):
    return (s.fillna('').astype(str).str.strip().str.strip('"').str.strip().str.lower()
              .str.replace(r'\s+',' ',regex=True))
clean = pd.DataFrame({
    'plant': df['Plant'].fillna('').astype(str).str.strip(),
    'light': norm(df['Light']).map(LIGHT_MAP),
    'watering': norm(df['Watering']).map(TARGET_MAP),
})
clean = clean.loc[(clean['plant']!='') & clean['light'].notna() & clean['watering'].notna()]
clean = clean.drop_duplicates().reset_index(drop=True)
print('Исходных строк:', len(df), '| уникальных пригодных строк:',len(clean),'| растений:',clean.plant.nunique())
display(clean.head(10))
display(clean.watering.value_counts().to_frame('Количество'))

## 5. Визуализация баланса классов

In [ ]:
fig,ax=plt.subplots(figsize=(7,4))
clean.watering.value_counts().plot(kind='bar',ax=ax)
ax.set(title='Режимы полива после очистки',xlabel='Режим',ylabel='Число записей')
plt.xticks(rotation=0);plt.tight_layout();plt.show()

## 6. Разделение train / test (75% / 25%)

In [ ]:
X=clean[['plant','light']]
y=clean['watering']
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=RANDOM_STATE,stratify=y)
print('Train:',len(X_train),'Test:',len(X_test))

## 7. Нормализация входа и обучение дерева решений
`OneHotEncoder` кодирует категориальные признаки в двоичную матрицу (0/1). `MinMaxScaler` здесь **не нужен**, поскольку числовых обучающих признаков нет. Все преобразования встроены в `Pipeline` во избежание утечки данных.

In [ ]:
preprocess = ColumnTransformer([('categories', OneHotEncoder(handle_unknown='ignore'), ['plant','light'])])
model=Pipeline([
    ('encode',preprocess),
    ('tree',DecisionTreeClassifier(max_depth=12,min_samples_leaf=3,class_weight='balanced',random_state=RANDOM_STATE))
])
model.fit(X_train,y_train)
print('Дерево обучено. Глубина:',model.named_steps['tree'].get_depth(),'Листьев:',model.named_steps['tree'].get_n_leaves())

## 8. Проверка качества на отложенной тестовой выборке

In [ ]:
pred=model.predict(X_test)
accuracy=float(accuracy_score(y_test,pred))
balanced=float(balanced_accuracy_score(y_test,pred))
print(f'Accuracy: {accuracy:.2%} | Balanced accuracy: {balanced:.2%}')
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,cmap='Blues',xticks_rotation=25)
plt.title('Матрица ошибок Decision Tree');plt.tight_layout();plt.show()
print('Ограничение: одинаковые названия растений могут присутствовать и в train, и в test. Для проверки на новых видах нужен отдельный тест с группировкой по plant.')

## 9. Визуализация дерева решений (первые 3 уровня)

In [ ]:
feature_names=model.named_steps['encode'].get_feature_names_out()
fig,ax=plt.subplots(figsize=(19,8))
plot_tree(model.named_steps['tree'],max_depth=3,feature_names=feature_names,
          class_names=model.named_steps['tree'].classes_,filled=True,rounded=True,fontsize=8,ax=ax)
ax.set_title('Дерево решений — первые 3 уровня');plt.tight_layout();plt.show()

## 10. Пробный прогноз

In [ ]:
example=pd.DataFrame([{'plant':clean.iloc[0].plant,'light':clean.iloc[0].light}])
print('Пример:',example.to_dict('records')[0])
print('Режим:',model.predict(example)[0])
print('Вероятности:', dict(zip(model.classes_,np.round(model.predict_proba(example)[0],3))))

## 11. Экспорт модели и очищенных данных для Streamlit
После выполнения скачайте `plant_model_for_streamlit.zip` и замените в репозитории соответствующие файлы внутри `model/` и `data/`.

In [ ]:
from google.colab import files
from zipfile import ZipFile, ZIP_DEFLATED
os.makedirs('model',exist_ok=True);os.makedirs('data',exist_ok=True)
joblib.dump(model,'model/watering_tree.joblib')
clean.to_csv('data/cleaned_for_training.csv',index=False)
metrics={'original_rows':int(len(df)),'usable_unique_rows':int(len(clean)), 'unique_plants':int(clean.plant.nunique()),
         'train_rows':int(len(X_train)), 'test_rows':int(len(X_test)), 'accuracy':accuracy,
         'balanced_accuracy':balanced,'class_counts':{str(k):int(v) for k,v in clean.watering.value_counts().items()},
         'caveats':'Only categorical watering regimes were learned. No numerical sensor data or actual next-watering times are in the dataset.'}
with open('model/metrics.json','w',encoding='utf-8') as f:json.dump(metrics,f,ensure_ascii=False,indent=2)
with ZipFile('plant_model_for_streamlit.zip','w',ZIP_DEFLATED) as z:
    for fn in ['model/watering_tree.joblib','model/metrics.json','data/cleaned_for_training.csv']:
        z.write(fn)
files.download('plant_model_for_streamlit.zip')
print('Модель готова для Streamlit!')